In [2]:
from datasets import Dataset
from pathlib import Path

def load_conll(filepath):
    with open(filepath, encoding="utf-8") as f:
        sentences = []
        labels = []
        current_tokens = []
        current_tags = []

        for line in f:
            line = line.strip()
            if not line:
                if current_tokens:
                    sentences.append(current_tokens)
                    labels.append(current_tags)
                    current_tokens, current_tags = [], []
                continue
            splits = line.split()
            if len(splits) >= 2:
                token = splits[0]
                tag = splits[-1]
                current_tokens.append(token)
                current_tags.append(tag)

        if current_tokens:
            sentences.append(current_tokens)
            labels.append(current_tags)

    return Dataset.from_dict({"tokens": sentences, "ner_tags": labels})

# 🗂️ Load from your labeled CoNLL file
dataset_path = "../data/labeled/sample_for_annotation.txt"
telegram_dataset = load_conll(dataset_path)
telegram_dataset = telegram_dataset.train_test_split(test_size=0.1)
telegram_dataset


DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 34
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 4
    })
})

In [3]:
from transformers import AutoTokenizer

model_checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

assert tokenizer.is_fast


In [4]:
label_list = ['B-Product', 'I-Product', 'B-LOC', 'I-LOC', 'B-PRICE', 'I-PRICE', 'O']
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)


{'B-Product': 0, 'I-Product': 1, 'B-LOC': 2, 'I-LOC': 3, 'B-PRICE': 4, 'I-PRICE': 5, 'O': 6}


In [5]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(example["tokens"],
                                 is_split_into_words=True,
                                 truncation=True,
                                 padding="max_length",
                                 max_length=128)

    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(label2id[example["ner_tags"][word_idx]])
        else:
            prev_tag = example["ner_tags"][word_idx]
            if prev_tag.startswith("B-"):
                label_ids.append(label2id[prev_tag.replace("B-", "I-")])
            else:
                label_ids.append(label2id[prev_tag])
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs


In [6]:
tokenized_dataset = telegram_dataset.map(tokenize_and_align_labels, batched=False)
tokenized_dataset["train"][0]


Map: 100%|██████████| 4/4 [00:00<00:00, 1413.06 examples/s]


{'tokens': ['አድራሻ',
  'ሜክሲኮ',
  'ኮሜርስ',
  'ጀርባ',
  'መዚድ',
  'ፕላዛ',
  'አንደኛ',
  'ደረጃ',
  'እንደወጡ',
  'ያገኙናል'],
 'ner_tags': ['O',
  'B-LOC',
  'I-LOC',
  'I-LOC',
  'I-LOC',
  'I-LOC',
  'I-LOC',
  'I-LOC',
  'I-LOC',
  'O'],
 'input_ids': [0,
  140042,
  93867,
  189512,
  18034,
  29768,
  25513,
  68980,
  238428,
  2370,
  23736,
  3236,
  6,
  32014,
  4712,
  13075,
  6,
  189017,
  32966,
  5193,
  87365,
  6,
  209387,
  52736,
  2,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1

In [7]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
import evaluate
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=-1)

    true_labels = [[id2label[label] for label in sent if label != -100]
                   for sent in labels]
    true_preds = [[id2label[pred] for pred, label in zip(sent_preds, sent_labels) if label != -100]
                  for sent_preds, sent_labels in zip(predictions, labels)]

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }


In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./ner_results",
    do_train=True,
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
)


In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)


In [11]:
trainer.train()


/Users/mikiyasegaye/MK_Lab/10 Academy/Amharic-E-commerce-Data-Extractor/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


TrainOutput(global_step=25, training_loss=0.962681884765625, metrics={'train_runtime': 10.795, 'train_samples_per_second': 15.748, 'train_steps_per_second': 2.316, 'total_flos': 11105614164480.0, 'train_loss': 0.962681884765625, 'epoch': 5.0})

In [12]:
model.save_pretrained("amharic-ner-model")
tokenizer.save_pretrained("amharic-ner-model")


('amharic-ner-model/tokenizer_config.json',
 'amharic-ner-model/special_tokens_map.json',
 'amharic-ner-model/tokenizer.json')

In [20]:
from transformers import pipeline
from pprint import pprint

model_path = "amharic-ner-model"
ner = pipeline("ner", model=model_path, tokenizer=model_path, aggregation_strategy="simple")

sample_texts = [
    "የወንድ ጫማ ጉርድ ሾላ",
    "የሴት ጅንስ ሱሪ አዲስ አበባ"
]

for idx, text in enumerate(sample_texts, 1):
    print(f"\n🔹 Sample {idx}: {text}")
    entities = ner(text)
    if not entities:
        print("  ⚠️ No entities found.")
    else:
        for ent in entities:
            print(f"  🟢 {ent['word']} → {ent['entity_group']} ({ent['score']:.2f})")


Device set to use mps:0



🔹 Sample 1: የወንድ ጫማ ጉርድ ሾላ
  ⚠️ No entities found.

🔹 Sample 2: የሴት ጅንስ ሱሪ አዲስ አበባ
  🟢 ሴት ጅንስ ሱሪ አዲስ → LOC (0.60)


In [21]:
from IPython.display import display, HTML

def highlight_entities(text, entities):
    """Visualize text with highlighted NER entities."""
    html = ""
    last_idx = 0
    for ent in sorted(entities, key=lambda x: x["start"]):
        # Add normal text before the entity
        html += text[last_idx:ent["start"]]
        # Add highlighted entity
        color = {
            "PRICE": "#ffd966",   # yellow
            "LOC": "#a4c2f4",     # blue
            "Product": "#b6d7a8", # green
        }.get(ent["entity_group"], "#e06666")  # fallback red
        html += f"<mark style='background-color: {color}; padding:2px; border-radius:3px;' title='{ent['entity_group']} ({ent['score']:.2f})'>{text[ent['start']:ent['end']]}</mark>"
        last_idx = ent["end"]
    html += text[last_idx:]  # Add any remaining text
    display(HTML(html))

# Example usage
sample_text = "በአዲስ አበባ ባለው ቦሌ በ 500 ብር የሚሸጥ የልጆች ጫማ"
entities = ner(sample_text)

highlight_entities(sample_text, entities)
